# BAB III — MODELING
## Penerapan Teknik SMOTE untuk Mengatasi Class Imbalance dalam Prediksi Customer Churn
### Menggunakan Algoritma Logistic Regression, Random Forest, dan XGBoost

---
**Program Studi Sistem Informasi — UPN "Veteran" Yogyakarta**
Disusun oleh: Zahirah Salsabila (124230164) | Amri Sabilly (124230147) | Faqih Aulia A.D. (124230150)
Sumber Dataset: [E-Commerce Customer Churn Dataset — Kaggle (Verma, 2021)](https://www.kaggle.com/datasets/ankitverma2010/ecommerce-customer-churn-analysis-and-prediction)

---
### Alur Kerja Notebook Ini

```
LOAD ARTIFACT DARI DATA PREPARATION
   │  (X_train_smote, y_train_smote, X_test_sc, y_test, scaler, dll)
   ▼
1. SETUP & DEFINISI MODEL
   │  ├─ Logistic Regression (baseline linear)
   │  ├─ Random Forest (ensemble tree-based)
   │  └─ XGBoost (gradient boosting)
   ▼
2. TRAINING (fit ke X_train_smote, y_train_smote — data SEIMBANG)
   ▼
3. PREDIKSI ke X_test_sc (data ASLI/imbalanced — representasi dunia nyata)
   ▼
4. EVALUASI per model
   │  ├─ Confusion Matrix
   │  ├─ Accuracy, Precision, Recall, F1-Score, ROC-AUC
   │  └─ Classification Report
   ▼
5. CROSS-VALIDATION (Stratified K-Fold pada data training)
   ▼
6. HYPERPARAMETER TUNING (GridSearchCV)
   ▼
7. PERBANDINGAN ANTAR MODEL (tabel + visualisasi)
   ▼
8. FEATURE IMPORTANCE (Random Forest & XGBoost)
   ▼
9. PEMILIHAN MODEL TERBAIK (berdasarkan F1-Score, bukan Accuracy)
   ▼
10. SIMPAN MODEL TERBAIK (.pkl) → untuk BAB Evaluation
```

> ⚠️ **Prinsip penting**: Training selalu menggunakan data **post-SMOTE** (`X_train_smote`, `y_train_smote`) agar model tidak bias ke kelas mayoritas. Evaluasi selalu menggunakan data **test asli** (`X_test_sc`, `y_test`) yang **tidak** ikut di-SMOTE, supaya hasil evaluasi mencerminkan performa model di dunia nyata — bukan performa di data sintetis.


---
## 0. Import Library

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 0 — Import Library
# ══════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
import sys
import subprocess
import importlib
import os
import joblib

pip_import_map = {
    "xgboost"          : "xgboost",
    "scikit-learn"     : "sklearn",
    "imbalanced-learn" : "imblearn",
    "matplotlib"       : "matplotlib",
    "seaborn"          : "seaborn",
    "joblib"           : "joblib",
}

for package, import_name in pip_import_map.items():
    try:
        importlib.import_module(import_name)
    except ModuleNotFoundError:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        except subprocess.CalledProcessError:
            # Beberapa environment (mis. Python "externally managed") butuh flag ini
            subprocess.check_call([sys.executable, "-m", "pip", "install", package, "--break-system-packages"])
    except Exception as err:
        if import_name == 'xgboost':
            print('⚠️ XGBoost import failed dengan kesalahan native library:')
            print(err)
            print('\nCatatan: Pada macOS, XGBoost memerlukan runtime OpenMP (`libomp`).')
            print('Pasang `libomp` dengan Homebrew: `brew install libomp`, lalu restart kernel.')
            raise
        raise

from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from xgboost                 import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

warnings.filterwarnings('ignore')

output_dir = 'public'
os.makedirs(output_dir, exist_ok=True)

# ── Gaya visual global (konsisten dengan notebook sebelumnya) ─────
plt.rcParams.update({
    'figure.dpi'      : 130,
    'axes.titlesize'  : 12,
    'axes.labelsize'  : 10,
    'xtick.labelsize' : 9,
    'ytick.labelsize' : 9,
    'legend.fontsize' : 9,
    'font.family'     : 'sans-serif',
})

C_LR     = '#4472C4'   # biru    → Logistic Regression
C_RF     = '#70AD47'   # hijau   → Random Forest
C_XGB    = '#ED7D31'   # oranye  → XGBoost
C_DANGER = '#C00000'   # merah   → peringatan/garis bantu

print('✅ Library berhasil diimport.')
print('   Versi scikit-learn :', __import__('sklearn').__version__)
print('   Versi xgboost      :', __import__('xgboost').__version__)


XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <4E82201A-ED82-3451-AD25-7886C77941A1> /Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/O

---
## 1. Load Artifact dari Data Preparation

Memuat seluruh output yang sudah disimpan di notebook **Data Preparation** (`output_smote/`). Tidak ada proses fit ulang di sini — `scaler`, `encoder_info`, dan `feature_columns` dimuat persis seperti saat training, untuk menjaga konsistensi dan mencegah data leakage.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 1 — Load Artifact dari Data Preparation
#  Ganti INPUT_DIR jika folder output_smote berada di lokasi lain
# ══════════════════════════════════════════════════════════════════
INPUT_DIR = './output_smote'      # <── sesuaikan path jika perlu

X_train_smote   = np.load(f'{INPUT_DIR}/X_train_smote.npy')
y_train_smote   = np.load(f'{INPUT_DIR}/y_train_smote.npy')
X_test_sc       = np.load(f'{INPUT_DIR}/X_test_sc.npy')
y_test          = np.load(f'{INPUT_DIR}/y_test.npy')

scaler           = joblib.load(f'{INPUT_DIR}/scaler.pkl')
encoder_info     = joblib.load(f'{INPUT_DIR}/encoder_info.pkl')
smote_params     = joblib.load(f'{INPUT_DIR}/smote_params.pkl')
feature_columns  = joblib.load(f'{INPUT_DIR}/feature_columns.pkl')

print('╔══════════════════════════════════════════════════════════════╗')
print('║              ARTIFACT BERHASIL DIMUAT                        ║')
print('╠══════════════════════════════════════════════════════════════╣')
print(f'║  X_train_smote  : {X_train_smote.shape}')
print(f'║  y_train_smote  : {y_train_smote.shape}')
print(f'║  X_test_sc      : {X_test_sc.shape}')
print(f'║  y_test         : {y_test.shape}')
print(f'║  Jumlah fitur   : {len(feature_columns)}')
print(f'║  Parameter SMOTE: {smote_params}')
print('╠══════════════════════════════════════════════════════════════╣')
print(f'║  Distribusi y_train_smote → Non-Churn: {(y_train_smote==0).sum():,} | Churn: {(y_train_smote==1).sum():,}  (seimbang ✅)')
print(f'║  Distribusi y_test         → Non-Churn: {(y_test==0).sum():,} | Churn: {(y_test==1).sum():,}  (asli, imbalanced)')
print('╚══════════════════════════════════════════════════════════════╝')


---
## 2. Definisi Model

Tiga algoritma yang dipakai mewakili tiga pendekatan berbeda:

| Model | Tipe | Karakteristik |
|---|---|---|
| **Logistic Regression** | Linear | Cepat, interpretable, jadi baseline. Mengasumsikan hubungan linear antar fitur & log-odds target. |
| **Random Forest** | Ensemble (Bagging) | Kumpulan decision tree, menangkap relasi non-linear, tahan terhadap overfitting dibanding single tree. |
| **XGBoost** | Ensemble (Boosting) | Tree dibangun bertahap, tiap tree baru memperbaiki kesalahan tree sebelumnya. Biasanya performa tertinggi di data tabular. |

Parameter awal memakai nilai yang umum dipakai sebagai titik awal yang wajar; optimasi lebih lanjut dilakukan di tahap Hyperparameter Tuning (Bagian 6).

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 2 — Definisi Model
# ══════════════════════════════════════════════════════════════════
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, max_depth=None, random_state=42, n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        eval_metric='logloss', random_state=42, n_jobs=-1
    ),
}

print('✅ 3 model berhasil didefinisikan:')
for name in models:
    print(f'   - {name}')


---
## 3. Training Model

Melatih ketiga model dengan `X_train_smote` dan `y_train_smote` — data training yang **sudah di-SMOTE dan di-scale** (seimbang 50:50). Model belajar pola dari data yang representatif untuk kedua kelas, bukan dari data yang didominasi kelas Non-Churn.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 3 — Training Model
#  Semua model dilatih pada X_train_smote (post-SMOTE, sudah di-scale)
# ══════════════════════════════════════════════════════════════════
trained_models = {}

print('⏳ Melatih model...\n')
for name, model in models.items():
    model.fit(X_train_smote, y_train_smote)
    trained_models[name] = model
    print(f'✅ {name:<22} selesai dilatih.')

print('\n📌 Semua model siap untuk evaluasi pada data test.')


---
## 4. Prediksi pada Data Test

**Poin krusial:** prediksi dilakukan pada `X_test_sc` — data test yang **tidak** ikut di-SMOTE, hanya di-scale. Proporsi kelasnya tetap asli (~16–17% Churn), sehingga hasil evaluasi mencerminkan performa model jika diterapkan ke data pelanggan riil, bukan performa semu di data sintetis.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 4 — Prediksi pada X_test_sc (data ASLI, bukan hasil SMOTE)
# ══════════════════════════════════════════════════════════════════
y_preds  = {}
y_probas = {}

for name, model in trained_models.items():
    y_preds[name]  = model.predict(X_test_sc)
    y_probas[name] = model.predict_proba(X_test_sc)[:, 1]   # probabilitas kelas Churn

print('✅ Prediksi selesai untuk semua model pada data test.')
print(f'   Jumlah sampel test       : {len(y_test):,}')
print(f'   Proporsi Churn aktual    : {(y_test==1).mean()*100:.1f}%  (representasi dunia nyata)')


---
## 5. Evaluasi Model

### 5.1 Metrik Evaluasi Lengkap

- **Accuracy** — persentase prediksi benar secara keseluruhan. **Tidak cukup diandalkan** pada data imbalanced (lihat simulasi Dummy Classifier di notebook Data Preparation).
- **Precision** — dari semua yang diprediksi Churn, berapa persen yang benar-benar Churn. Penting jika biaya *false alarm* (treat pelanggan loyal sebagai berisiko churn) cukup mahal.
- **Recall** — dari semua pelanggan yang benar-benar Churn, berapa persen berhasil terdeteksi. **Metrik terpenting** untuk kasus churn, karena gagal mendeteksi pelanggan yang akan churn = kehilangan kesempatan retensi.
- **F1-Score** — harmonic mean Precision & Recall, jadi metrik penyeimbang utama yang digunakan untuk membandingkan model.
- **ROC-AUC** — mengukur kemampuan model membedakan kelas Churn vs Non-Churn di seluruh kemungkinan threshold (tidak terpaku di satu titik cutoff 0.5).

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 5.1 — Hitung Seluruh Metrik Evaluasi
# ══════════════════════════════════════════════════════════════════
results = []

for name in trained_models:
    y_pred  = y_preds[name]
    y_proba = y_probas[name]

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)
    auc  = roc_auc_score(y_test, y_proba)

    results.append({
        'Model'     : name,
        'Accuracy'  : acc,
        'Precision' : prec,
        'Recall'    : rec,
        'F1-Score'  : f1,
        'ROC-AUC'   : auc,
    })

results_df = pd.DataFrame(results).set_index('Model')
results_df_pct = (results_df * 100).round(2)

print('╔══════════════════════════════════════════════════════════════╗')
print('║              RINGKASAN METRIK EVALUASI (%)                   ║')
print('╚══════════════════════════════════════════════════════════════╝')
display(results_df_pct)


### 5.2 Classification Report per Model

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 5.2 — Classification Report Detail per Model
# ══════════════════════════════════════════════════════════════════
for name in trained_models:
    print(f'═'*65)
    print(f' {name}')
    print(f'═'*65)
    print(classification_report(
        y_test, y_preds[name], target_names=['Non-Churn', 'Churn']
    ))


### 5.3 Confusion Matrix

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 5.3 — Visualisasi Confusion Matrix (3 model berdampingan)
# ══════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Confusion Matrix — Perbandingan 3 Model', fontsize=14, fontweight='bold', y=1.03)

for ax, name in zip(axes, trained_models):
    cm = confusion_matrix(y_test, y_preds[name])
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
        xticklabels=['Non-Churn', 'Churn'], yticklabels=['Non-Churn', 'Churn'],
        annot_kws={'fontsize': 13, 'fontweight': 'bold'}
    )
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('Prediksi')
    ax.set_ylabel('Aktual')

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'model_01_confusion_matrix.png'), bbox_inches='tight', dpi=150)
plt.show()


### 5.4 ROC Curve

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 5.4 — ROC Curve Perbandingan 3 Model
# ══════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(7, 6))
colors = {'Logistic Regression': C_LR, 'Random Forest': C_RF, 'XGBoost': C_XGB}

for name in trained_models:
    fpr, tpr, _ = roc_curve(y_test, y_probas[name])
    auc_val = roc_auc_score(y_test, y_probas[name])
    ax.plot(fpr, tpr, label=f'{name} (AUC = {auc_val:.3f})',
            color=colors[name], linewidth=2.2)

ax.plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=1.2, label='Random Guess (AUC = 0.500)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Perbandingan 3 Model', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'model_02_roc_curve.png'), bbox_inches='tight', dpi=150)
plt.show()


---
## 6. Cross-Validation (Stratified K-Fold)

Cross-validation dilakukan pada data **training** (`X_train_smote`, `y_train_smote`) untuk memastikan performa model **stabil** di berbagai pembagian data, bukan kebetulan bagus pada satu split tertentu. `StratifiedKFold` memastikan proporsi kelas tetap terjaga di setiap fold.

**Kegunaan:** memberi bukti *robustness* model — kalau standar deviasi antar fold kecil, model dianggap stabil dan tidak terlalu sensitif terhadap variasi data.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 6 — Stratified K-Fold Cross-Validation (k=5) pada F1-Score
# ══════════════════════════════════════════════════════════════════
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = {}
print('⏳ Menjalankan 5-Fold Cross-Validation (metrik: F1-Score)...\n')
for name, model in models.items():
    scores = cross_val_score(model, X_train_smote, y_train_smote, cv=skf, scoring='f1', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:<22} → F1 per fold: {np.round(scores, 3)}')
    print(f'{"":22}   Mean = {scores.mean():.4f}  |  Std = {scores.std():.4f}\n')

cv_summary = pd.DataFrame({
    name: [scores.mean(), scores.std()] for name, scores in cv_results.items()
}, index=['Mean F1', 'Std F1']).T

print('Ringkasan Cross-Validation:')
display(cv_summary.round(4))


---
## 7. Hyperparameter Tuning (GridSearchCV)

Mencari kombinasi parameter terbaik untuk **setiap model** menggunakan `GridSearchCV` dengan metrik scoring **F1-Score** (bukan accuracy, karena dataset imbalanced di sisi evaluasi nyata).

> ⚠️ Catatan: grid di bawah dibuat ringkas agar cepat dijalankan. Untuk eksplorasi lebih dalam (skripsi), grid bisa diperluas, tapi waktu komputasi akan bertambah signifikan — pertimbangkan `RandomizedSearchCV` jika grid terlalu besar.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 7.1 — Hyperparameter Tuning: Logistic Regression
# ══════════════════════════════════════════════════════════════════
param_grid_lr = {
    'C'       : [0.01, 0.1, 1, 10],
    'penalty' : ['l2'],
    'solver'  : ['lbfgs'],
}

grid_lr = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    param_grid_lr, cv=5, scoring='f1', n_jobs=-1
)
grid_lr.fit(X_train_smote, y_train_smote)

print('✅ Logistic Regression — Best Params:', grid_lr.best_params_)
print(f'   Best CV F1-Score : {grid_lr.best_score_:.4f}')


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 7.2 — Hyperparameter Tuning: Random Forest
# ══════════════════════════════════════════════════════════════════
param_grid_rf = {
    'n_estimators' : [100, 200],
    'max_depth'    : [None, 10, 20],
    'min_samples_split' : [2, 5],
}

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid_rf, cv=5, scoring='f1', n_jobs=-1
)
grid_rf.fit(X_train_smote, y_train_smote)

print('✅ Random Forest — Best Params:', grid_rf.best_params_)
print(f'   Best CV F1-Score : {grid_rf.best_score_:.4f}')


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 7.3 — Hyperparameter Tuning: XGBoost
# ══════════════════════════════════════════════════════════════════
param_grid_xgb = {
    'n_estimators'  : [100, 200],
    'max_depth'     : [4, 6, 8],
    'learning_rate' : [0.05, 0.1],
}

grid_xgb = GridSearchCV(
    XGBClassifier(eval_metric='logloss', random_state=42, n_jobs=-1),
    param_grid_xgb, cv=5, scoring='f1', n_jobs=-1
)
grid_xgb.fit(X_train_smote, y_train_smote)

print('✅ XGBoost — Best Params:', grid_xgb.best_params_)
print(f'   Best CV F1-Score : {grid_xgb.best_score_:.4f}')


### 7.4 Re-train Model dengan Parameter Terbaik

Melatih ulang ketiga model menggunakan parameter terbaik dari `GridSearchCV`, lalu mengevaluasinya kembali pada `X_test_sc` untuk melihat apakah tuning benar-benar meningkatkan performa di data test.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 7.4 — Re-train dengan Parameter Terbaik & Evaluasi Ulang
# ══════════════════════════════════════════════════════════════════
tuned_models = {
    'Logistic Regression': grid_lr.best_estimator_,
    'Random Forest'      : grid_rf.best_estimator_,
    'XGBoost'             : grid_xgb.best_estimator_,
}

tuned_results = []
y_preds_tuned  = {}
y_probas_tuned = {}

for name, model in tuned_models.items():
    y_pred  = model.predict(X_test_sc)
    y_proba = model.predict_proba(X_test_sc)[:, 1]
    y_preds_tuned[name]  = y_pred
    y_probas_tuned[name] = y_proba

    tuned_results.append({
        'Model'     : name,
        'Accuracy'  : accuracy_score(y_test, y_pred),
        'Precision' : precision_score(y_test, y_pred, zero_division=0),
        'Recall'    : recall_score(y_test, y_pred, zero_division=0),
        'F1-Score'  : f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC'   : roc_auc_score(y_test, y_proba),
    })

tuned_results_df = pd.DataFrame(tuned_results).set_index('Model')

print('Perbandingan F1-Score: Sebelum Tuning vs Sesudah Tuning')
comparison_tuning = pd.DataFrame({
    'Sebelum Tuning' : results_df['F1-Score'],
    'Sesudah Tuning' : tuned_results_df['F1-Score'],
})
comparison_tuning['Δ Perubahan'] = comparison_tuning['Sesudah Tuning'] - comparison_tuning['Sebelum Tuning']
display((comparison_tuning * 100).round(2).rename(columns=lambda c: c if 'Δ' not in c else c + ' (poin %)'))


---
## 8. Perbandingan Antar Model (Final, Setelah Tuning)

Menyusun seluruh metrik dari model yang sudah ditala (*tuned*) ke dalam satu tabel ringkas dan satu visualisasi, untuk menjawab pertanyaan utama: **model mana yang paling efektif untuk prediksi customer churn dengan SMOTE?**

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 8.1 — Tabel Perbandingan Final
# ══════════════════════════════════════════════════════════════════
final_results_df = tuned_results_df.copy()
final_results_pct = (final_results_df * 100).round(2)

print('╔══════════════════════════════════════════════════════════════╗')
print('║         TABEL PERBANDINGAN FINAL ANTAR MODEL (%)              ║')
print('╚══════════════════════════════════════════════════════════════╝')
display(final_results_pct.style.highlight_max(axis=0, color='#C6E0B4'))


In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 8.2 — Visualisasi Bar Chart Perbandingan Metrik
# ══════════════════════════════════════════════════════════════════
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
x = np.arange(len(metrics_to_plot))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
colors_list = [C_LR, C_RF, C_XGB]

for i, name in enumerate(final_results_df.index):
    values = final_results_df.loc[name, metrics_to_plot].values * 100
    bars = ax.bar(x + (i - 1) * width, values, width, label=name, color=colors_list[i], edgecolor='white')
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                f'{v:.1f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylabel('Skor (%)')
ax.set_ylim(0, 110)
ax.set_title('Perbandingan Metrik Evaluasi Antar Model (Setelah Tuning)', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'model_03_perbandingan_metrik.png'), bbox_inches='tight', dpi=150)
plt.show()


---
## 9. Feature Importance

Random Forest dan XGBoost menyediakan atribut `feature_importances_` yang menunjukkan kontribusi relatif setiap fitur terhadap keputusan model. Ini memberi **insight bisnis**: fitur apa yang paling memengaruhi keputusan pelanggan untuk churn.

(Logistic Regression tidak punya `feature_importances_`, tapi koefisiennya — `coef_` — bisa diinterpretasikan serupa untuk model linear; ditampilkan juga sebagai pembanding.)

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 9 — Feature Importance: Random Forest, XGBoost, & Koefisien LR
# ══════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# ── Random Forest ──
rf_importance = pd.Series(
    tuned_models['Random Forest'].feature_importances_, index=feature_columns
).sort_values(ascending=False).head(10)
axes[0].barh(rf_importance.index[::-1], rf_importance.values[::-1], color=C_RF)
axes[0].set_title('Top 10 Feature Importance\nRandom Forest', fontsize=11)
axes[0].set_xlabel('Importance Score')

# ── XGBoost ──
xgb_importance = pd.Series(
    tuned_models['XGBoost'].feature_importances_, index=feature_columns
).sort_values(ascending=False).head(10)
axes[1].barh(xgb_importance.index[::-1], xgb_importance.values[::-1], color=C_XGB)
axes[1].set_title('Top 10 Feature Importance\nXGBoost', fontsize=11)
axes[1].set_xlabel('Importance Score')

# ── Logistic Regression (koefisien absolut) ──
lr_coef = pd.Series(
    np.abs(tuned_models['Logistic Regression'].coef_[0]), index=feature_columns
).sort_values(ascending=False).head(10)
axes[2].barh(lr_coef.index[::-1], lr_coef.values[::-1], color=C_LR)
axes[2].set_title('Top 10 |Koefisien|\nLogistic Regression', fontsize=11)
axes[2].set_xlabel('Magnitude Koefisien')

for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'model_04_feature_importance.png'), bbox_inches='tight', dpi=150)
plt.show()

print('📌 Fitur paling berpengaruh (Random Forest):')
print(rf_importance.head(5).to_string())
print()
print('📌 Fitur paling berpengaruh (XGBoost):')
print(xgb_importance.head(5).to_string())


---
## 10. Pemilihan Model Terbaik

Model terbaik dipilih berdasarkan **F1-Score** sebagai metrik utama (menyeimbangkan Precision & Recall), dengan **Recall** sebagai pertimbangan kedua — karena dalam konteks churn, gagal mendeteksi pelanggan yang akan churn (*false negative*) umumnya lebih merugikan secara bisnis dibanding salah memprediksi pelanggan loyal sebagai berisiko (*false positive*).

**Bukan accuracy** — karena (seperti dibuktikan dengan simulasi Dummy Classifier di notebook Data Preparation) accuracy tinggi bisa dicapai tanpa mendeteksi churn sama sekali.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 10 — Pemilihan Model Terbaik berdasarkan F1-Score
# ══════════════════════════════════════════════════════════════════
best_model_name = final_results_df['F1-Score'].idxmax()
best_model      = tuned_models[best_model_name]
best_metrics    = final_results_df.loc[best_model_name]

print('╔══════════════════════════════════════════════════════════════╗')
print('║                  MODEL TERBAIK TERPILIH                      ║')
print('╠══════════════════════════════════════════════════════════════╣')
print(f'║  Model       : {best_model_name}')
print(f'║  Accuracy    : {best_metrics["Accuracy"]*100:.2f}%')
print(f'║  Precision   : {best_metrics["Precision"]*100:.2f}%')
print(f'║  Recall      : {best_metrics["Recall"]*100:.2f}%')
print(f'║  F1-Score    : {best_metrics["F1-Score"]*100:.2f}%  ← metrik utama')
print(f'║  ROC-AUC     : {best_metrics["ROC-AUC"]*100:.2f}%')
print('╠══════════════════════════════════════════════════════════════╣')
print('║  Alasan pemilihan:')
print('║  • F1-Score tertinggi → keseimbangan Precision & Recall terbaik')
print('║  • Lebih relevan dari Accuracy untuk data churn yang imbalanced')
print('╚══════════════════════════════════════════════════════════════╝')


---
## 11. Simpan Model Terbaik

Model terbaik disimpan sebagai artifact `.pkl` untuk dipakai di **BAB Evaluation** (atau tahap deployment), beserta metadata penting (nama model, parameter terbaik, dan metrik evaluasi), tanpa perlu melakukan training ulang.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 11 — Simpan Model Terbaik & Metadata
# ══════════════════════════════════════════════════════════════════
output_model_dir = 'output_model'
os.makedirs(output_model_dir, exist_ok=True)

joblib.dump(best_model, f'{output_model_dir}/best_model.pkl')

metadata = {
    'model_name'       : best_model_name,
    'best_params'      : best_model.get_params(),
    'metrics'          : best_metrics.to_dict(),
    'feature_columns'  : feature_columns,
    'smote_params'     : smote_params,
}
joblib.dump(metadata, f'{output_model_dir}/model_metadata.pkl')

# Simpan juga seluruh model (bukan hanya yang terbaik) untuk dokumentasi/perbandingan
for name, model in tuned_models.items():
    safe_name = name.lower().replace(' ', '_')
    joblib.dump(model, f'{output_model_dir}/{safe_name}.pkl')

# Simpan tabel hasil evaluasi final sebagai CSV
final_results_pct.to_csv(f'{output_model_dir}/final_evaluation_results.csv')

print('✅ Artifact model berhasil disimpan di folder output_model/:')
for f in sorted(os.listdir(output_model_dir)):
    size = os.path.getsize(f'{output_model_dir}/{f}') / 1024
    print(f'   {f:<35}  {size:.1f} KB')

print()
print('📦 Artifact siap dipakai di notebook Evaluation:')
print('   - best_model.pkl              → model terbaik, siap predict langsung')
print('   - model_metadata.pkl          → nama model, parameter, & metrik')
print('   - logistic_regression.pkl     → model LR (tuned)')
print('   - random_forest.pkl           → model RF (tuned)')
print('   - xgboost.pkl                 → model XGB (tuned)')
print('   - final_evaluation_results.csv→ tabel metrik final')


---
## 12. Checklist Verifikasi & Ringkasan

Sebelum lanjut ke **BAB Evaluation** (atau penulisan kesimpulan), pastikan seluruh poin di bawah ini terpenuhi.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 12 — Checklist Verifikasi Akhir Modeling
# ══════════════════════════════════════════════════════════════════
checks = []

# 1. Semua model berhasil dilatih
checks.append(('Ketiga model (LR, RF, XGBoost) berhasil dilatih',
               len(trained_models) == 3))

# 2. Prediksi dilakukan pada data test ASLI (bukan hasil SMOTE)
test_churn_pct = (y_test == 1).mean() * 100
checks.append((f'Evaluasi memakai y_test asli (proporsi Churn ≈ asli, aktual {test_churn_pct:.1f}%)',
               5 <= test_churn_pct <= 30))

# 3. Semua metrik tersedia untuk setiap model
checks.append(('Tabel hasil evaluasi (Accuracy, Precision, Recall, F1, ROC-AUC) lengkap',
               final_results_df.shape == (3, 5)))

# 4. Cross-validation sudah dijalankan
checks.append(('Cross-validation (5-Fold) sudah dijalankan untuk semua model',
               len(cv_results) == 3))

# 5. Hyperparameter tuning sudah dilakukan
checks.append(('Hyperparameter tuning (GridSearchCV) sudah dilakukan untuk semua model',
               all(name in tuned_models for name in models)))

# 6. Model terbaik berhasil ditentukan
checks.append((f'Model terbaik berhasil ditentukan: {best_model_name}',
               best_model_name in tuned_models))

# 7. Artifact tersimpan
required_files = ['best_model.pkl', 'model_metadata.pkl', 'final_evaluation_results.csv']
for fname in required_files:
    checks.append((f'Artifact tersimpan: {fname}',
                   os.path.exists(f'{output_model_dir}/{fname}')))

print('╔══════════════════════════════════════════════════════════════╗')
print('║            CHECKLIST VERIFIKASI MODELING                      ║')
print('╠══════════════════════════════════════════════════════════════╣')
all_passed = True
for desc, passed in checks:
    icon = '✅' if passed else '❌'
    if not passed:
        all_passed = False
    print(f'║  {icon}  {desc}')
print('╠══════════════════════════════════════════════════════════════╣')
if all_passed:
    print('║  🎉 SEMUA CHECK LOLOS — siap lanjut ke BAB Evaluation!         ')
else:
    print('║  ⚠️  ADA CHECK YANG GAGAL — periksa kembali sebelum lanjut     ')
print('╚══════════════════════════════════════════════════════════════╝')
print()
print('📌 Ringkasan akhir:')
print(f'   Model terbaik   : {best_model_name}')
print(f'   F1-Score        : {best_metrics["F1-Score"]*100:.2f}%')
print(f'   ROC-AUC         : {best_metrics["ROC-AUC"]*100:.2f}%')
print(f'   Artifact model  : output_model/best_model.pkl')
